In [1]:
import pandas as pd

In [2]:
is_blast = pd.read_csv("blast_output_isfinder.tsv", sep="\t", header=None)

In [3]:
is_blast.columns = ["qseqid", "sseqid" ,"pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore", "qlen", "slen"]

In [6]:
is_blast_filter = is_blast[is_blast['length']/is_blast['slen'] > 0.9]
is_blast_filter = is_blast_filter[is_blast_filter['pident'] > 90]

print(len(set(is_blast_filter.qseqid)))
print(len(is_blast_filter))

838
4943


In [59]:
unique_records = list(set(is_blast_filter.qseqid))

In [60]:
def ranges_overlap(start1, end1, start2, end2):
    """
    Check if two ranges overlap.

    Args:
        start1, end1: Start and end coordinates of first range
        start2, end2: Start and end coordinates of second range

    Returns:
        bool: True if ranges overlap, False otherwise
    """
    # Ranges overlap if: start1 <= end2 AND start2 <= end1
    return start1 <= end2 and start2 <= end1
filtered_ise = []
for record in unique_records:
    is_elements = []
    record_ise = is_blast_filter[is_blast_filter['qseqid'] == record]

    is_elements.append(record_ise.iloc[0])

    for _,hit in record_ise.iterrows():
        overlap = False
        for ise in is_elements:
            if ranges_overlap(ise.qstart, ise.qend, hit.qstart, hit.qend):
                overlap = True
                break

        if not overlap:
            is_elements.append(hit)


    filtered_ise.append(is_elements)
filtered_ise = sum(filtered_ise, [])

In [61]:
filtered_ise = pd.DataFrame(filtered_ise)

filtered_ise.to_csv("filtered_isfinder.csv", index=True, header=True)

In [62]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import numpy as np

# Output file for sequences with IS elements removed
output_file = "O-antigen_db.reverse_complement.is_removed.fasta"

# Open output file for writing
with open(output_file, 'w') as out_handle:
    # Iterate through all sequences in the file
    for record in SeqIO.parse("O-antigen_db.fasta_all.reverse_complement.fasta", "fasta"):
        seq_id = record.id
        seq_str = str(record.seq)

        # Get IS elements for this sequence from dataframe
        is_regions = filtered_ise[filtered_ise['qseqid'] == seq_id][['qstart', 'qend']].values

        if len(is_regions) > 0:
            print("Processing {}: found {} IS element(s)".format(seq_id, len(is_regions)))

            # Convert to 0-based indexing and sort
            is_regions = np.array([[int(start)-1, int(end)-1] for start, end in is_regions])
            is_regions = is_regions[is_regions[:, 0].argsort()]

            # Build new sequence without IS elements
            regions_to_keep = []
            last_end = 0

            for start, end in is_regions:
                # Add region before this IS element
                if start > last_end:
                    regions_to_keep.append(seq_str[last_end:start])

                # Update last_end to be after this IS element
                last_end = max(last_end, end + 1)

            # Add remaining region after last IS element
            if last_end < len(seq_str):
                regions_to_keep.append(seq_str[last_end:])

            # Create new sequence
            new_seq_str = ''.join(regions_to_keep)

            # Create new SeqRecord with same header
            new_record = SeqRecord(
                Seq(new_seq_str),
                id=record.id,
                description=record.description,
                name=record.name
            )

        else:
            # No IS elements, use original record
            new_record = record

        # Write to output file
        SeqIO.write(new_record, out_handle, 'fasta')

print("Done! Output written to: {}".format(output_file))

Processing NZ_CP183114.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP080056.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP034106.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP043183.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP139953.1_reverse_complement: found 2 IS element(s)
Processing NZ_CP062247.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP013952.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP093221.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP134389.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP018965.1_reverse_complement: found 1 IS element(s)
Processing NZ_CM125433.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP091712.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP102378.1_reverse_complement: found 1 IS element(s)
Processing NZ_CM125679.1_reverse_complement: found 1 IS element(s)
Processing NZ_CP100941.1_reverse_complement: found 1 IS elemen

11059